## Demonstrating the Born Rule

<img src="images/Born-rule.png" width="250" align="left"/>

A qubit is described by a normalized vector in a two-dimensional complex Hilbert space with computational basis states $|0\rangle$ and $|1\rangle$. Every pure qubit state can be visualized as a point on the Bloch sphere and expressed in column-vector and ket notation as

$$
|S\rangle =
\begin{pmatrix}
\cos\left(\frac{\theta}{2}\right) \\
e^{i\varphi}\sin\left(\frac{\theta}{2}\right)
\end{pmatrix}
=
\cos\left(\frac{\theta}{2}\right)|0\rangle
+
e^{i\varphi}\sin\left(\frac{\theta}{2}\right)|1\rangle,
\qquad
0 \leq \theta \leq \pi,\quad
0 \leq \varphi < 2\pi.
$$

The **Born rule** states that the probability of each measurement outcome equals the squared magnitude of the corresponding probability amplitude. For the qubit

$$
|S\rangle = a|0\rangle + b|1\rangle,
\qquad
a=\cos\left(\frac{\theta}{2}\right),\quad
b=e^{i\varphi}\sin\left(\frac{\theta}{2}\right),
$$

where the amplitudes satisfy the normalization condition $|a|^2+|b|^2=1$, the probability of obtaining the classical measurement outcome **0** is $P(0)=|a|^2$, while the probability of obtaining the outcome **1** is $P(1)=|b|^2$.
<br clear="left"/>

### Task

Write a Qiskit program that experimentally demonstrates the Born rule.

Your program should:

1. Generate random values of the Bloch-sphere angles $\theta$ and $\varphi$.
2. Prepare the corresponding qubit state
   $$
   |S\rangle=
   \cos\left(\frac{\theta}{2}\right)|0\rangle+
   e^{i\varphi}\sin\left(\frac{\theta}{2}\right)|1\rangle.
   $$
3. Display the Bloch vector representing the generated state.
4. Compute the theoretical Born probabilities
   $$
   P(0)=\cos^2\left(\frac{\theta}{2}\right),\qquad
   P(1)=\sin^2\left(\frac{\theta}{2}\right).
   $$
5. Measure the qubit repeatedly using several different numbers of shots (for example, 10, 100, 1000, and 10000).
6. Display the measured probability distributions together with the theoretical distribution predicted by the Born rule.
7. Observe how the measured probabilities converge to the theoretical values as the number of measurements increases.

### Expected Output

The program should display:

- the randomly generated values of $\theta$ and $\varphi$;
- the quantum circuit that prepares the qubit;
- the Bloch sphere showing the generated quantum state;
- the theoretical probabilities $P(0)$ and $P(1)$;
- a combined probability-distribution plot comparing the experimental results for different numbers of shots with the theoretical Born distribution.

The experimental distributions should approach the theoretical probabilities as the number of measurements increases, illustrating the statistical nature of quantum measurement predicted by the Born rule.


### Optional Challenge

Repeat the experiment several times and compare the results for different randomly generated qubit states.

Investigate how the measurement statistics depend on the polar angle $\theta$ and explain why changing the azimuthal angle $\varphi$ does not affect the probabilities measured in the computational basis.


In [ ]:
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit.visualization import (
    plot_bloch_vector,
    plot_distribution,
    circuit_drawer
)
from qiskit_aer import AerSimulator


# ============================================================
# 1. Generate random angles
# ============================================================

rng = np.random.default_rng()

theta = rng.uniform(0, np.pi)
phi = rng.uniform(0, 2 * np.pi)

print("Randomly generated angles")
print("-------------------------")
print(f"θ = {theta:.4f} rad = {np.degrees(theta):.2f}°")
print(f"φ = {phi:.4f} rad = {np.degrees(phi):.2f}°")


# ============================================================
# 2. Prepare the qubit state
#
# |S⟩ = cos(θ/2)|0⟩ + e^(iφ) sin(θ/2)|1⟩
# ============================================================

qc = QuantumCircuit(1)

qc.ry(theta, 0)
qc.rz(phi, 0)


# ============================================================
# 3. Obtain the statevector
# ============================================================

state = Statevector(qc)

a0 = state.data[0]
a1 = state.data[1]

print("\nStatevector")
print("-----------")
print(f"a₀ = {a0:.4f}")
print(f"a₁ = {a1:.4f}")

print(
    f"\n|S⟩ = ({a0:.4f})|0⟩ "
    f"+ ({a1:.4f})|1⟩"
)


# ============================================================
# 4. Display the Bloch vector
#
# x = sin(θ)cos(φ)
# y = sin(θ)sin(φ)
# z = cos(θ)
# ============================================================

x = np.sin(theta) * np.cos(phi)
y = np.sin(theta) * np.sin(phi)
z = np.cos(theta)

bloch_vector = [x, y, z]

print("\nBloch-vector coordinates")
print("------------------------")
print(f"x = {x:.4f}")
print(f"y = {y:.4f}")
print(f"z = {z:.4f}")

bloch_figure = plot_bloch_vector(
    bloch_vector,
    title="Randomly generated qubit state"
)

display(bloch_figure)
plt.close(bloch_figure)


# ============================================================
# 5. Calculate the theoretical Born probabilities
#
# p(0) = |a₀|²
# p(1) = |a₁|²
# ============================================================

p0 = float(abs(a0) ** 2)
p1 = float(abs(a1) ** 2)

theoretical_distribution = {
    "0": p0,
    "1": p1
}

print("\nBorn probabilities")
print("------------------")
print(f"p(0) = |a₀|² = {p0:.6f}")
print(f"p(1) = |a₁|² = {p1:.6f}")
print(f"p(0) + p(1) = {p0 + p1:.6f}")


# ============================================================
# 6. Create the measurement circuit
# ============================================================

qc.measure_all()
print("\nQuantum circuit:")
display(circuit_drawer(qc, output="mpl"))


# ============================================================
# 7. Perform measurements for different shot numbers
# ============================================================

simulator = AerSimulator()

shot_values = [10, 100, 1000, 10000]

distributions = []
legend = []

for shots in shot_values:

    result = simulator.run(
        qc,
        shots=shots
    ).result()

    counts = result.get_counts()

    distributions.append(counts)
    legend.append(f"{shots} shots")

    measured_p0 = counts.get("0", 0) / shots
    measured_p1 = counts.get("1", 0) / shots

    print(f"\nMeasurements: {shots} shots")
    print(f"Observed frequency of 0: {measured_p0:.6f}")
    print(f"Observed frequency of 1: {measured_p1:.6f}")


# ============================================================
# 8. Add the theoretical distribution
# ============================================================

distributions.append(theoretical_distribution)
legend.append("Theoretical")


# ============================================================
# 9. Display all distributions in one grouped diagram
# ============================================================

colors = [
    "lightblue",
    "cornflowerblue",
    "royalblue",
    "navy",
    "crimson"
]

figure = plot_distribution(
    distributions,
    legend=legend,
    color=colors,
    title="Experimental demonstration of the Born rule",
    bar_labels=True,
    figsize=(11, 6)
)

display(figure)
plt.close(figure)